# Fine-tuned RoBERTa — Error Analysis

Identifies and samples prediction errors from `bert_predictions.csv` for manual inspection.

**Pipeline:**
1. Reconstruct test split (test_size=0.2, random_state=42)
2. Separate errors into False Positives and False Negatives
3. Per-topic error rate breakdown
4. Sample errors for manual reading

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

## 1. Load data and reconstruct test split

In [2]:
df = pd.read_csv("bert_predictions.csv", low_memory=False)

# Filter out rating=3 (no label)
df_valid = df[df["label"].notna()].copy()
df_valid["label"] = df_valid["label"].astype(int)
df_valid["pred"]  = (df_valid["pred_sentiment"] == "positive").astype(int)

# Reconstruct test split
_, df_test = train_test_split(
    df_valid, test_size=0.2, random_state=42, stratify=df_valid["label"]
)

print(f"Test set size: {len(df_test)}")
print(f"Correct:   {(df_test['label'] == df_test['pred']).sum()}")
print(f"Incorrect: {(df_test['label'] != df_test['pred']).sum()}")

Test set size: 24849
Correct:   24148
Incorrect: 701


## 2. Separate error types

- **False Positive (FP):** model predicted *positive*, true label is *negative*
- **False Negative (FN):** model predicted *negative*, true label is *positive*

In [3]:
errors = df_test[df_test["label"] != df_test["pred"]].copy()

fp = errors[(errors["label"] == 0) & (errors["pred"] == 1)]  # predicted pos, actually neg
fn = errors[(errors["label"] == 1) & (errors["pred"] == 0)]  # predicted neg, actually pos

print(f"Total errors:     {len(errors)} ({len(errors)/len(df_test)*100:.1f}% error rate)")
print(f"False Positives:  {len(fp)}  (negative review predicted as positive)")
print(f"False Negatives:  {len(fn)}  (positive review predicted as negative)")

Total errors:     701 (2.8% error rate)
False Positives:  358  (negative review predicted as positive)
False Negatives:  343  (positive review predicted as negative)


## 3. Per-topic error rate

In [4]:
df_test_topic = df_test[df_test["topic_label"].notna()].copy()

topic_stats = df_test_topic.groupby("topic_label").apply(
    lambda g: pd.Series({
        "total":      len(g),
        "errors":     (g["label"] != g["pred"]).sum(),
        "fp":         ((g["label"] == 0) & (g["pred"] == 1)).sum(),
        "fn":         ((g["label"] == 1) & (g["pred"] == 0)).sum(),
        "error_rate": (g["label"] != g["pred"]).mean(),
    })
).sort_values("error_rate", ascending=False)

topic_stats["total"]  = topic_stats["total"].astype(int)
topic_stats["errors"] = topic_stats["errors"].astype(int)
topic_stats["fp"]     = topic_stats["fp"].astype(int)
topic_stats["fn"]     = topic_stats["fn"].astype(int)

print(f"{'Topic':<30} {'Total':>6} {'Errors':>7} {'FP':>5} {'FN':>5} {'Error%':>8}")
print("-" * 65)
for topic, row in topic_stats.iterrows():
    print(f"{topic:<30} {int(row['total']):>6,} {int(row['errors']):>7,} "
          f"{int(row['fp']):>5,} {int(row['fn']):>5,} {row['error_rate']:>7.1%}")

Topic                           Total  Errors    FP    FN   Error%
-----------------------------------------------------------------
Shipping damage                 1,117      79    42    37    7.1%
Customer service / returns      1,628     111    51    60    6.8%
String quality                  1,068      53    26    27    5.0%
Electronics / controls            415      19    11     8    4.6%
Fret / neck setup               2,390     107    51    56    4.5%
Visual appearance                 758      27    13    14    3.6%
Accessories                     1,779      58    26    32    3.3%
Guitar size                     1,059      34    21    13    3.2%
Tuning stability                1,036      27    10    17    2.6%
Playability / chords            1,083      18     9     9    1.7%
Beginner learning               2,579      39    24    15    1.5%
Pickups                         1,477      22    10    12    1.5%
Setup / action                    991      13     8     5    1.3%
Acoustic 

/var/folders/lw/bfntj97n2fvf9_3gn6x4td980000gn/T/ipykernel_9759/2461258197.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  topic_stats = df_test_topic.groupby("topic_label").apply(


## 4. Sample errors for manual inspection

Read these carefully and look for patterns:  
sarcasm, mixed sentiment, very short reviews, domain-specific language, etc.

In [5]:
SAMPLE_N = 20  # number of samples per error type
RANDOM_STATE = 42

cols_to_show = ["rating", "topic_label", "label", "pred", "pred_pos_prob", "text"]

fp_sample = fp[cols_to_show].sample(min(SAMPLE_N, len(fp)), random_state=RANDOM_STATE)
fn_sample = fn[cols_to_show].sample(min(SAMPLE_N, len(fn)), random_state=RANDOM_STATE)

print(f"Sampled {len(fp_sample)} False Positives and {len(fn_sample)} False Negatives")

Sampled 20 False Positives and 20 False Negatives


In [6]:
# --- FALSE POSITIVES: negative reviews misclassified as positive ---
# Look for: sarcasm, reviews that start positive but end negative,
# complaints buried under polite language

print("=" * 80)
print("FALSE POSITIVES — negative reviews predicted as positive")
print("=" * 80)

for i, (_, row) in enumerate(fp_sample.iterrows(), 1):
    print(f"\n[{i}] Rating: {row['rating']} | Topic: {row['topic_label']} | Confidence: {row['pred_pos_prob']:.3f}")
    print("-" * 60)
    print(row["text"][:600])  # show first 600 characters
    if len(str(row["text"])) > 600:
        print("[... truncated]")

FALSE POSITIVES — negative reviews predicted as positive

[1] Rating: 2 | Topic: Beginner learning | Confidence: 0.999
------------------------------------------------------------
Guitar works, but is a beginner guitar...didnt realize that.  Im more advanced and was looking for a certain type.  As it is, good guitar.for a beginner.

[2] Rating: 1 | Topic: Customer service / returns | Confidence: 0.985
------------------------------------------------------------
This is a true beginner guitar. It’s beautiful and has a wonderful sound. The problem I have is the tuner was sent without a battery! I have been to four different stores looking for that size battery, and no luck! I will be returning this!

[3] Rating: 2 | Topic: Fret / neck setup | Confidence: 0.970
------------------------------------------------------------
The action on the fret not as good as I hoped and sound quality is fair.

[4] Rating: 2 | Topic: Electronics / controls | Confidence: 0.918
------------------------------

In [7]:
# --- FALSE NEGATIVES: positive reviews misclassified as negative ---
# Look for: mixed reviews with minor complaints, very short reviews,
# reviews that focus on a negative aspect but overall positive

print("=" * 80)
print("FALSE NEGATIVES — positive reviews predicted as negative")
print("=" * 80)

for i, (_, row) in enumerate(fn_sample.iterrows(), 1):
    print(f"\n[{i}] Rating: {row['rating']} | Topic: {row['topic_label']} | Confidence: {row['pred_pos_prob']:.3f}")
    print("-" * 60)
    print(row["text"][:600])
    if len(str(row["text"])) > 600:
        print("[... truncated]")

FALSE NEGATIVES — positive reviews predicted as negative

[1] Rating: 5 | Topic: Beginner learning | Confidence: 0.017
------------------------------------------------------------
plays like butta

[2] Rating: 5 | Topic: Accessories | Confidence: 0.096
------------------------------------------------------------
Everything is amazing with this guitar!!<br />But, the amp is absolute junk! It ONLY operates on battery and you CANNOT use headphones due to the WRONG aux jack installed.<br />Doesnt matter anyway, it's become a Ghost portal!👻

[3] Rating: 4 | Topic: Customer service / returns | Confidence: 0.048
------------------------------------------------------------
I have it, but I have failed to follow through with using it... I believe it will work for me when Im ready

[4] Rating: 4 | Topic: Beginner learning | Confidence: 0.033
------------------------------------------------------------
It's a cheap guitar so expect your money's worth. However, as a beginner, it's better to start 

## 5. Errors by confidence level

Low-confidence errors (pred_pos_prob near 0.5) are expected — the model was unsure.  
High-confidence errors are more interesting: the model was wrong AND certain.

In [8]:
bins   = [0.0, 0.3, 0.5, 0.7, 0.9, 1.0]
labels = ["0.0-0.3", "0.3-0.5", "0.5-0.7", "0.7-0.9", "0.9-1.0"]

errors["conf_bin"] = pd.cut(errors["pred_pos_prob"], bins=bins, labels=labels)

conf_breakdown = errors.groupby("conf_bin", observed=True).size().reset_index(name="n_errors")
conf_breakdown["pct"] = conf_breakdown["n_errors"] / len(errors) * 100

print(f"{'Confidence':>12} {'Errors':>8} {'%':>8}")
print("-" * 32)
for _, row in conf_breakdown.iterrows():
    print(f"{str(row['conf_bin']):>12} {int(row['n_errors']):>8,} {row['pct']:>7.1f}%")

print(f"\nHigh-confidence errors (>0.9 or <0.1):")
high_conf_errors = errors[
    (errors["pred_pos_prob"] > 0.9) | (errors["pred_pos_prob"] < 0.1)
]
print(f"  {len(high_conf_errors)} errors ({len(high_conf_errors)/len(errors)*100:.1f}% of all errors)")

  Confidence   Errors        %
--------------------------------
     0.0-0.3      322    45.9%
     0.3-0.5       21     3.0%
     0.5-0.7       13     1.9%
     0.7-0.9       36     5.1%
     0.9-1.0      309    44.1%

High-confidence errors (>0.9 or <0.1):
  589 errors (84.0% of all errors)


In [9]:
# Sample high-confidence errors — most interesting for error analysis
print("=" * 80)
print("HIGH-CONFIDENCE ERRORS (model was wrong but very certain)")
print("=" * 80)

hc_sample = high_conf_errors[cols_to_show].sample(
    min(15, len(high_conf_errors)), random_state=RANDOM_STATE
)

for i, (_, row) in enumerate(hc_sample.iterrows(), 1):
    true_label = "positive" if row["label"] == 1 else "negative"
    pred_label = "positive" if row["pred"]  == 1 else "negative"
    print(f"\n[{i}] Rating: {row['rating']} | Topic: {row['topic_label']}")
    print(f"     True: {true_label} | Predicted: {pred_label} | Confidence: {row['pred_pos_prob']:.3f}")
    print("-" * 60)
    print(row["text"][:600])
    if len(str(row["text"])) > 600:
        print("[... truncated]")

HIGH-CONFIDENCE ERRORS (model was wrong but very certain)

[1] Rating: 1 | Topic: Shipping damage
     True: negative | Predicted: positive | Confidence: 0.979
------------------------------------------------------------
El amplificador es demasiado pequeno y suena muy mal, es demasiado pesada para un niño de 8 años

[2] Rating: 4 | Topic: nan
     True: positive | Predicted: negative | Confidence: 0.025
------------------------------------------------------------
My friend has this guitar, and I have played it multiple times.  The aesthetics of the guitar are obviously cool.  The hot pink glitter makes it a very bold instrument.<br /><br />However, there is very poor resonance and tone with this.  It is made smaller for girls, but I found the neck width to be borderline too small for me (and I have smaller hands).  The depth of the neck is nice, makes it much easier to wrap your thumb around.<br /><br />I guess the resonance is sacrificed with the sparkles.  Because of its coating, th